# Orbit Wars Benchmark (Colab)\n\nThis notebook benchmarks parity and approximation backends, including GPU-oriented JAX throughput.

In [ ]:
# Colab setup (uncomment if needed)\n# !pip install -U kaggle-environments jax jaxlib pandas matplotlib

In [ ]:
import jax\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom orbit_wars_fast.benchmark import (
    run_benchmark,
    print_benchmark,
    benchmark_dataframe,
    run_fleet_cap_sweep,
    recommend_fleet_cap,
)
\nprint('JAX device:', jax.devices())

In [ ]:
results = run_benchmark(
    num_games=20,
    max_steps=300,
    seed=1337,
    include_kaggle_runtime=True,
    jax_num_envs=1024,
    jax_steps=3000,
    include_numba_variant=True,
    include_parallel_parity=True,
    parallel_workers=8,
    include_jax_toggle_sweep=True,
    jax_max_fleets=1024,
)
print_benchmark(results)\ndf = benchmark_dataframe(results)\ndf

In [ ]:
# Sweep JAX env batch size to study GPU scaling\nsweep = []\nfor n in [128, 256, 512, 1024, 2048]:\n    r = run_benchmark(\n        num_games=5,\n        max_steps=150,\n        seed=1337,\n        include_kaggle_runtime=False,\n        jax_num_envs=n,\n        jax_steps=2000,\n        include_numba_variant=False,\n        include_parallel_parity=False,\n    )\n    row = r['jax core (approx)'].copy()\n    row['jax_num_envs'] = n\n    sweep.append(row)\n\nsweep_df = pd.DataFrame(sweep)\nsweep_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))\nax[0].plot(sweep_df['jax_num_envs'], sweep_df['ms_per_step'], marker='o')\nax[0].set_title('JAX core latency per step')\nax[0].set_xlabel('jax_num_envs')\nax[0].set_ylabel('ms_per_step')\n\nax[1].plot(sweep_df['jax_num_envs'], sweep_df['env_steps_per_s'], marker='o')\nax[1].set_title('JAX core throughput')\nax[1].set_xlabel('jax_num_envs')\nax[1].set_ylabel('env_steps_per_s')\n\nplt.tight_layout()
plt.show()

In [ ]:
# Fleet-cap sweep + recommendation\ncap_rows = run_fleet_cap_sweep(\n    caps=[256, 384, 512, 768, 1024, 1536, 2048],\n    jax_num_envs=1024,\n    jax_steps=3000,\n    seed=1337,\n    use_swept_collision=False,\n    use_segment_sun_check=False,\n)\ncap_df = pd.DataFrame(cap_rows)\ncap_df

In [ ]:
recommend_fleet_cap(\n    cap_rows,\n    utilization_target=0.70,\n    overflow_step_rate_target=0.0,\n    safety_margin=1.30,\n)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))\nax[0].plot(cap_df['max_fleets'], cap_df['ms_per_step'], marker='o')\nax[0].set_title('Cap vs latency')\nax[0].set_xlabel('max_fleets')\nax[0].set_ylabel('ms_per_step')\n\nax[1].plot(cap_df['max_fleets'], cap_df['fleet_active_max_utilization'], marker='o')\nax[1].set_title('Cap utilization')\nax[1].set_xlabel('max_fleets')\nax[1].set_ylabel('fleet_active_max_utilization')\n\nplt.tight_layout()\nplt.show()